In [2]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [4]:
# recreate the document objects
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),
    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),
    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),
    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.""" 
    ), metadata={"source": "Doc4"}),
]

In [5]:
# Create a FAISS vector store from the documents
embedding_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = FAISS.from_documents(documents=docs, embedding=embedding_model)

In [6]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [7]:
# Set up the compressor using an LLM
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
compressor = LLMChainExtractor.from_llm(llm=llm)

In [9]:
# Create the contextual compression retriever
contextual_compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [10]:
# query the retriever
query = "What is photosynthesis?"
compressed_results = contextual_compression_retriever.invoke(query)

In [11]:
for i, doc in enumerate(compressed_results):
    print(f"\n ---Result {i+1} ---")
    print(doc.page_content)


 ---Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

 ---Result 2 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.
